# Label CafeF 2023 With DeepSeek V4 Flash

Notebook này đọc 1.000 dòng đầu từ `data/raw/cafef_raw/cafef_news_2023.csv`, nạp `system_prompt.txt` và `few_shots.txt`, rồi gọi DeepSeek Chat Completions để trích xuất sự kiện tài chính theo schema JSON.

Cách xử lý few-shot trong notebook này:
- `system_prompt.txt` giữ toàn bộ schema/quy tắc ổn định trong role `system`.
- `few_shots.txt` được đưa vào một message `user` ổn định ngay trước bài báo hiện tại. Cách này hợp với file few-shot đang là markdown/rút gọn, và giúp phần prefix lặp lại dễ được context cache.
- Bài báo hiện tại nằm ở message `user` cuối cùng, mỗi request chỉ xử lý một bài để tránh lẫn entity/event giữa các bài.
- API bật `response_format={"type": "json_object"}` để ép output là JSON hợp lệ theo hướng dẫn DeepSeek.

Nguồn API chính thức:
- Chat completions: https://api-docs.deepseek.com/api/create-chat-completion
- JSON output: https://api-docs.deepseek.com/guides/json_mode
- Models/pricing/cache fields: https://api-docs.deepseek.com/quick_start/pricing


In [1]:
from __future__ import annotations

import json
import os
import time
import urllib.error
import urllib.request
from pathlib import Path
from typing import Any

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent.parent

DATA_PATH = ROOT / "data/raw/cafef_raw/cafef_news_2023.csv"
SYSTEM_PROMPT_PATH = ROOT / "src/LLM_labeling/system_prompt.txt"
FEW_SHOTS_PATH = ROOT / "src/LLM_labeling/few_shots.txt"
OUTPUT_PATH = ROOT / "data/processed/cafef_2023_first1000.jsonl"

MODEL = "deepseek-v4-flash"
BASE_URL = "https://api.deepseek.com"
N_ROWS = 1000
REQUEST_SLEEP_SECONDS = 0.15
MAX_TOKENS = 8192
TEMPERATURE = 0
DISABLE_THINKING = True

print("ROOT:", ROOT)
print("DATA_PATH exists:", DATA_PATH.exists())
print("SYSTEM_PROMPT_PATH exists:", SYSTEM_PROMPT_PATH.exists())
print("FEW_SHOTS_PATH exists:", FEW_SHOTS_PATH.exists())


ROOT: /Users/buithianhdao/SE365
DATA_PATH exists: True
SYSTEM_PROMPT_PATH exists: True
FEW_SHOTS_PATH exists: True


In [2]:
system_prompt = SYSTEM_PROMPT_PATH.read_text(encoding="utf-8").strip()
few_shots = FEW_SHOTS_PATH.read_text(encoding="utf-8").strip()

df = pd.read_csv(DATA_PATH).head(N_ROWS).copy()
df["article_id"] = df["article_id"].astype(str)

print("Rows:", len(df))
print("System prompt chars:", len(system_prompt))
print("Few-shot chars:", len(few_shots))
display(df[["article_id", "title", "summary", "published_date", "url"]].head(3))


Rows: 1000
System prompt chars: 15124
Few-shot chars: 8440


,article_id,title,summary,published_date,url
0,188231229103257892,"Thủy điện Hủa Na (HNA) ước lãi 217 tỷ đồng, lê...","Ngày 12/01/2024, hơn 235 triệu cổ phiếu HNA sẽ...",2023-12-29,https://cafef.vn/thuy-dien-hua-na-hna-uoc-lai-...
1,188231229094645055,Hành trình 16 năm và những dấu ấn tiên phong c...,Hành trình 16 năm từng bước chinh phục khách h...,2023-12-29,https://cafef.vn/hanh-trinh-16-nam-va-nhung-da...
2,188231229082554747,"UBCKNN xử phạt Công ty Đại Nam của ông Dũng ""L...",Công ty cổ phần Đại Nam bị phạt vì có hành vi ...,2023-12-29,https://cafef.vn/ubcknn-xu-phat-cong-ty-dai-na...


## Prompt Builders

`few_shots.txt` hiện là markdown ví dụ rút gọn, không phải các cặp `user`/`assistant` hoàn chỉnh. Vì vậy notebook dùng nó như một khối tham khảo ổn định. Nếu sau này bạn tạo few-shot đầy đủ dạng `input/output`, có thể đổi sang alternating messages: `user(example_input)`, `assistant(example_output)`, rồi mới tới bài thật.

In [3]:
def clean_text(value: Any) -> str:
    if pd.isna(value):
        return ""
    return str(value).strip()


def build_article_prompt(row: pd.Series) -> str:
    return f"""Bây giờ hãy trích xuất bài báo sau theo đúng schema JSON trong system prompt.

Yêu cầu bắt buộc:
- Chỉ trả về JSON hợp lệ, không markdown, không giải thích.
- `doc_id` nếu cần thì dùng article_id bên dưới.
- Nếu không có sự kiện tài chính rõ ràng: events = [] và main_topic = "other".

article_id: {clean_text(row.get('article_id'))}
source: {clean_text(row.get('source'))}
category: {clean_text(row.get('category'))}

TIÊU ĐỀ:
{clean_text(row.get('title'))}

TÓM TẮT:
{clean_text(row.get('summary'))}

NỘI DUNG:
{clean_text(row.get('content'))}
"""


def build_messages(row: pd.Series, include_few_shots: bool = True) -> list[dict[str, str]]:
    messages = [{"role": "system", "content": system_prompt}]
    if include_few_shots and few_shots:
        messages.append({
            "role": "user",
            "content": (
                "Dưới đây là các ví dụ mẫu. Hãy học cách chọn event_type, main_topic, "
                "cách điền attributes/context/evidence_text. Không trích xuất lại các ví dụ này.\n\n"
                f"{few_shots}"
            ),
        })
    messages.append({"role": "user", "content": build_article_prompt(row)})
    return messages


def extract_json_object(text: str) -> dict[str, Any]:
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}")
        if start >= 0 and end > start:
            return json.loads(text[start:end + 1])
        raise


preview_messages = build_messages(df.iloc[0])
print("Number of messages:", len(preview_messages))
for i, msg in enumerate(preview_messages):
    print(i, msg["role"], len(msg["content"]), msg["content"][:160].replace("\n", " "))


Number of messages: 3
0 system 15124 Bạn là hệ thống trích xuất sự kiện tài chính chuyên nghiệp cho thị trường chứng khoán Việt Nam. Nhiệm vụ của bạn là đọc MỘT bài báo tài chính tiếng Việt và tríc
1 user 8590 Dưới đây là các ví dụ mẫu. Hãy học cách chọn event_type, main_topic, cách điền attributes/context/evidence_text. Không trích xuất lại các ví dụ này.  ### Ví dụ 
2 user 2564 Bây giờ hãy trích xuất bài báo sau theo đúng schema JSON trong system prompt.  Yêu cầu bắt buộc: - Chỉ trả về JSON hợp lệ, không markdown, không giải thích. - `


In [4]:
def deepseek_chat(messages: list[dict[str, str]], *, max_retries: int = 3) -> dict[str, Any]:
    api_key = os.environ.get("DEEPSEEK_API_KEY")
    if not api_key:
        raise RuntimeError("Missing DEEPSEEK_API_KEY. Set it before running API cells.")

    payload: dict[str, Any] = {
        "model": MODEL,
        "messages": messages,
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
        "response_format": {"type": "json_object"},
        "stream": False,
    }
    if DISABLE_THINKING:
        payload["thinking"] = {"type": "disabled"}

    body = json.dumps(payload, ensure_ascii=False).encode("utf-8")
    req = urllib.request.Request(
        f"{BASE_URL}/chat/completions",
        data=body,
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        },
        method="POST",
    )

    for attempt in range(max_retries):
        try:
            with urllib.request.urlopen(req, timeout=180) as resp:
                return json.loads(resp.read().decode("utf-8"))
        except urllib.error.HTTPError as e:
            error_body = e.read().decode("utf-8", errors="replace")
            if e.code in {429, 500, 502, 503, 504} and attempt < max_retries - 1:
                time.sleep(2 ** attempt)
                continue
            raise RuntimeError(f"DeepSeek HTTP {e.code}: {error_body}") from e
        except urllib.error.URLError:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
                continue
            raise

    raise RuntimeError("DeepSeek request failed after retries")


def label_one(row: pd.Series, include_few_shots: bool = True) -> dict[str, Any]:
    messages = build_messages(row, include_few_shots=include_few_shots)
    response = deepseek_chat(messages)
    choice = response["choices"][0]
    content = choice["message"].get("content") or ""

    parsed: dict[str, Any] | None = None
    parse_error: str | None = None
    try:
        parsed = extract_json_object(content)
        parsed.setdefault("doc_id", clean_text(row.get("article_id")))
    except json.JSONDecodeError as e:
        # Keep raw_response/usage so malformed JSON can be inspected or repaired later.
        parse_error = repr(e)

    return {
        "article_id": clean_text(row.get("article_id")),
        "title": clean_text(row.get("title")),
        "published_date": clean_text(row.get("published_date")),
        "url": clean_text(row.get("url")),
        "label": parsed,
        "raw_response": content,
        "finish_reason": choice.get("finish_reason"),
        "usage": response.get("usage", {}),
        "model": response.get("model", MODEL),
        "response_id": response.get("id"),
        "error": parse_error,
    }


# Nhập DeepSeek API key

In [5]:
import os
import getpass

os.environ.pop("DEEPSEEK_API_KEY", None)
os.environ["DEEPSEEK_API_KEY"] = getpass.getpass("Nhập lại DeepSeek API key: ").strip()


## Test 1 Article

Chạy cell này trước để xem API key, JSON mode, prompt và schema có ổn không. Để tránh gọi API ngoài ý muốn, mặc định `RUN_ONE_API_CALL = False`.

In [6]:
RUN_ONE_API_CALL = False

if RUN_ONE_API_CALL:
    one_result = label_one(df.iloc[0], include_few_shots=True)
    print(json.dumps(one_result["label"], ensure_ascii=False, indent=2))
    print("usage:", one_result["usage"])
else:
    print("Dry run only. Set RUN_ONE_API_CALL = True to call DeepSeek for the first article.")
    print(build_article_prompt(df.iloc[0])[:1200])


Dry run only. Set RUN_ONE_API_CALL = True to call DeepSeek for the first article.
Bây giờ hãy trích xuất bài báo sau theo đúng schema JSON trong system prompt.

Yêu cầu bắt buộc:
- Chỉ trả về JSON hợp lệ, không markdown, không giải thích.
- `doc_id` nếu cần thì dùng article_id bên dưới.
- Nếu không có sự kiện tài chính rõ ràng: events = [] và main_topic = "other".

article_id: 188231229103257892
source: cafef
category: Thị trường chứng khoán

TIÊU ĐỀ:
Thủy điện Hủa Na (HNA) ước lãi 217 tỷ đồng, lên sàn HoSE ngay trong tháng 1/2024

TÓM TẮT:
Ngày 12/01/2024, hơn 235 triệu cổ phiếu HNA sẽ chính thức được giao dịch trên sàn HSX với giá tham chiếu trong ngày giao dịch đầu tiên là 18.350 đồng/cổ phiếu.

NỘI DUNG:
HNA:
Ngày 28/12/2023, Công ty Cổ phần Chứng khoán Rồng Việt (Rồng Việt) đã phối hợp cùng Công ty Cổ phần Thủy điện Hủa Na (Mã CK: HNA) tổ chức Hội nghị nhà đầu tư. Buổi hội thảo thu hút sự tham gia của các chuyên gia phân tích, nhân viên môi giới và đông đảo các Nhà đầu tư chứng kh

In [7]:
print(df.iloc[0]['content'] )


HNA:
Ngày 28/12/2023, Công ty Cổ phần Chứng khoán Rồng Việt (Rồng Việt) đã phối hợp cùng Công ty Cổ phần Thủy điện Hủa Na (Mã CK: HNA) tổ chức Hội nghị nhà đầu tư. Buổi hội thảo thu hút sự tham gia của các chuyên gia phân tích, nhân viên môi giới và đông đảo các Nhà đầu tư chứng khoán.
Tại hội thảo, Ban lãnh đạo Công ty Cổ phần Thủy điện Hủa Na đã giới thiệu tổng quan về Công ty cũng như kế ước tính kết quả kinh doanh năm 2023.
Theo đó, trong năm 2023, tổng sản lượng điện thương mại của Hủa Nam ước đạt 557.860mWh, doanh thu thuần ước tính 755 tỷ đồng và lợi nhuận sau thuế 217 tỷ đồng. Công ty hiện đang có kế hoạch tìm kiếm, nghiên cứu các dự án thủy điện quy mô vừa và nhỏ tại Nghệ An, Thanh Hóa, Tây Nguyên. Ngoài ra, Hủa Na cũng nghiên cứu thực hiện dự án điện mặt trời ở lòng hồ thủy điện Hủa Na.
Hiện tại, Hủa Na đang vận hành nhà máy điện Hủa Na với giá bán điện bình quân 1.186 đồng/kWh.
Tại hội thảo, đại diện Thủy điện Hủa Na cũng chia sẻ với các nhà đầu tư về việc niêm yết cổ phiếu 

## Batch Label First 1000 Rows

Output được append vào JSONL. Nếu notebook dừng giữa chừng, chạy lại cell này sẽ bỏ qua `article_id` đã có trong output.

In [8]:
RUN_BATCH = True

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

processed_ids: set[str] = set()
# Chỉ skip các article_id đã label thành công. Các dòng lỗi sẽ được retry.
if OUTPUT_PATH.exists():
    with OUTPUT_PATH.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                record = json.loads(line)
                if record.get("label") is not None and not record.get("error"):
                    processed_ids.add(str(record.get("article_id")))

todo = df.loc[~df["article_id"].isin(processed_ids)].copy()
print("Output:", OUTPUT_PATH)
print("Already processed:", len(processed_ids))
print("Todo:", len(todo))

if RUN_BATCH:
    with OUTPUT_PATH.open("a", encoding="utf-8") as out:
        for n, (_, row) in enumerate(todo.iterrows(), start=1):
            article_id = clean_text(row.get("article_id"))
            try:
                result = label_one(row, include_few_shots=True)
                result.setdefault("error", None)
            except Exception as e:
                result = {
                    "article_id": article_id,
                    "title": clean_text(row.get("title")),
                    "published_date": clean_text(row.get("published_date")),
                    "url": clean_text(row.get("url")),
                    "label": None,
                    "raw_response": None,
                    "finish_reason": None,
                    "usage": {},
                    "model": MODEL,
                    "error": repr(e),
                }
            out.write(json.dumps(result, ensure_ascii=False) + "\n")
            out.flush()
            if n % 10 == 0 or result.get("error"):
                print(f"{n}/{len(todo)} article_id={article_id} error={result.get('error')}")
            time.sleep(REQUEST_SLEEP_SECONDS)
else:
    print("Dry run only. Set RUN_BATCH = True to label the first 1000 rows.")


Output: /Users/buithianhdao/SE365/data/processed/cafef_2023_first1000.jsonl
Already processed: 0
Todo: 1000
10/1000 article_id=188231228133628241 error=None
20/1000 article_id=188231227185121636 error=None
30/1000 article_id=188231227103309926 error=None
40/1000 article_id=188231225160441405 error=None
50/1000 article_id=18823122509020593 error=None
60/1000 article_id=188231224121555926 error=None
70/1000 article_id=188231221002530769 error=None
80/1000 article_id=188231222133550946 error=None
90/1000 article_id=188231221191212787 error=None
100/1000 article_id=18823121909024836 error=None
110/1000 article_id=188231220150404752 error=None
120/1000 article_id=18823121710424306 error=None
130/1000 article_id=188231216231645356 error=None
140/1000 article_id=188231218141828788 error=None
150/1000 article_id=188231214183448573 error=None
160/1000 article_id=188231214075029919 error=None
170/1000 article_id=1882312150942061 error=None
180/1000 article_id=188231212215557139 error=None
190/10

## Inspect Results And Cost Tokens

In [9]:
def load_jsonl(path: Path) -> list[dict[str, Any]]:
    if not path.exists():
        return []
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


records = load_jsonl(OUTPUT_PATH)
print("records:", len(records))
if records:
    result_df = pd.DataFrame(records)
    result_df["raw_response_chars"] = result_df["raw_response"].apply(lambda x: len(x) if isinstance(x, str) else 0)
    display(result_df[["article_id", "title", "published_date", "error", "finish_reason", "raw_response_chars"]].head(20))

    usage = pd.json_normalize([r.get("usage", {}) for r in records])
    if not usage.empty:
        print(usage.sum(numeric_only=True).to_string())

    labels = [r.get("label") for r in records if r.get("label")]
    event_counts = {}
    for label in labels:
        for event in label.get("events", []) or []:
            event_type = event.get("event_type")
            event_counts[event_type] = event_counts.get(event_type, 0) + 1
    print("event_counts:")
    print(json.dumps(dict(sorted(event_counts.items(), key=lambda x: str(x[0]))), ensure_ascii=False, indent=2))


records: 1000


,article_id,title,published_date,error,finish_reason,raw_response_chars
0,188231229103257892,"Thủy điện Hủa Na (HNA) ước lãi 217 tỷ đồng, lê...",2023-12-29,None,stop,2695
1,188231229094645055,Hành trình 16 năm và những dấu ấn tiên phong c...,2023-12-29,None,stop,750
2,188231229082554747,"UBCKNN xử phạt Công ty Đại Nam của ông Dũng ""L...",2023-12-29,None,stop,1797
3,188231228162951823,Từng gây chấn động với giá gần 1 triệu đồng/cp...,2023-12-29,None,stop,3582
4,188231228183151294,Góc nhìn CTCK: Rung lắc có thể sớm xuất hiện k...,2023-12-28,None,stop,1831
5,18823122817101572,"Cùng chiều khối ngoại, tự doanh CTCK mua ròng ...",2023-12-28,None,stop,3122
6,188231228165707197,Cổ phiếu nhà Vingroup tiếp tục gây chú ý,2023-12-28,None,stop,5124
7,188231228164957233,Hãng dược hơn trăm tuổi Nhật Bản tung 200 tỷ m...,2023-12-28,None,stop,3003
8,18823122815555081,Khối ngoại tiếp đà mua ròng 330 tỷ đồng trong ...,2023-12-28,None,stop,16297
9,188231228133628241,Cổ phiếu của một công ty vốn nghìn tỷ bị huỷ n...,2023-12-28,None,stop,1762


prompt_tokens                          10531217
completion_tokens                       1426644
total_tokens                           11957861
prompt_cache_hit_tokens                 9181184
prompt_cache_miss_tokens                1350033
prompt_tokens_details.cached_tokens     9181184
event_counts:
{
  "declare_bankruptcy": 6,
  "dividend": 270,
  "earnings_report": 399,
  "equity_freeze": 7,
  "equity_overweight": 236,
  "equity_repurchase": 9,
  "equity_underweight": 311,
  "lawsuit": 20,
  "legal_penalty": 110,
  "macro_indicator": 145,
  "merge_org": 12,
  "other": 36,
  "personnel_change": 76,
  "price_movement": 999,
  "share_issuance": 209,
  "transfer_money": 47,
  "transfer_ownership": 60
}


## Inspect Raw Response For Failed Articles

Sau khi batch chạy, dùng helper này để in `raw_response` của một `article_id`, nhất là các dòng bị `JSONDecodeError`.

In [11]:
import re


def show_raw_response(article_id: str, *, full: bool = False, context_chars: int = 1200) -> None:
    records = load_jsonl(OUTPUT_PATH)
    matches = [r for r in records if str(r.get("article_id")) == str(article_id)]
    print("matches:", len(matches))
    if not matches:
        return

    for idx, record in enumerate(matches, start=1):
        raw = record.get("raw_response")
        error = record.get("error")
        print("\n--- match", idx, "---")
        print("title:", record.get("title"))
        print("error:", error)
        print("finish_reason:", record.get("finish_reason"))
        print("usage:", record.get("usage"))
        print("raw_response_chars:", len(raw) if isinstance(raw, str) else None)

        if not isinstance(raw, str):
            print("raw_response is None. Record này có thể được tạo trước khi sửa notebook; hãy chạy lại batch cho article này.")
            continue

        if full:
            print(raw)
            continue

        match = re.search(r"char (\d+)", str(error))
        if match:
            pos = int(match.group(1))
            start = max(0, pos - context_chars)
            end = min(len(raw), pos + context_chars)
            print(f"showing raw_response[{start}:{end}] around char {pos}:")
            print(raw[start:end])
        else:
            print(f"showing raw_response[:{context_chars}]:")
            print(raw[:context_chars])


# Example:
show_raw_response("188231220091258805")
show_raw_response("188231220091258805", full=True)


matches: 1

--- match 1 ---
title: Lịch sự kiện và tin vắn chứng khoán ngày 20/12
error: None
finish_reason: stop
usage: {'prompt_tokens': 10496, 'completion_tokens': 4903, 'total_tokens': 15399, 'prompt_tokens_details': {'cached_tokens': 10368}, 'prompt_cache_hit_tokens': 10368, 'prompt_cache_miss_tokens': 128}
raw_response_chars: 13559
showing raw_response[:1200]:
{
  "doc_id": "188231220091258805",
  "language": "vi",
  "main_topic": "ownership_change",
  "entities": [
    {"name": "PYN ELITE FUND (NON-UCITS)", "type": "Organization", "ticker": null, "role_in_article": "subject"},
    {"name": "Công ty Cổ phần Dịch vụ Hàng hóa Sài Gòn", "type": "Company", "ticker": "SCS", "role_in_article": "mentioned"},
    {"name": "CT TNHH Sài Gòn 3 Jean", "type": "Organization", "ticker": null, "role_in_article": "subject"},
    {"name": "Công ty Cổ phần Pin Ắc quy miền Nam", "type": "Company", "ticker": "PAC", "role_in_article": "mentioned"},
    {"name": "Nguyễn Thị Nga", "type": "Person", "ti